In [2]:
#Task 14: LangGraph
#Implementing LangGraph React agent
# ---------------------------
# Imports
# ---------------------------
from typing import TypedDict, List
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, END

# ---------------------------
# 1. Define State
# ---------------------------
class AgentState(TypedDict):
    messages: List[BaseMessage]

# ---------------------------
# 2. LLaMA 3 Model
# ---------------------------
llm = ChatOllama(model="llama3")

# ---------------------------
# 3. Simple Tool (Calculator)
# ---------------------------
def calculator(expression: str) -> str:
    try:
        return str(eval(expression))
    except Exception:
        return "Error in calculation"

# ---------------------------
# 4. Agent Node (LLM)
# ---------------------------
def agent_node(state: AgentState):
    messages = state["messages"]

    system_prompt = """
You are a ReAct agent.

Follow this format strictly:

Thought: think about the problem
Action: calculator
Action Input: expression

OR

Final Answer: result
"""

    response = llm.invoke([HumanMessage(content=system_prompt)] + messages)
    return {"messages": messages + [response]}

# ---------------------------
# 5. Tool Node
# ---------------------------
def tool_node(state: AgentState):
    last_message = state["messages"][-1].content

    if "Action:" in last_message and "calculator" in last_message:
        try:
            expression = last_message.split("Action Input:")[-1].strip()
            result = calculator(expression)
        except Exception:
            result = "Failed"

        return {
            "messages": state["messages"] + [
                AIMessage(content=f"Observation: {result}")
            ]
        }

    return state

# ---------------------------
# 6. Router
# ---------------------------
def should_continue(state: AgentState):
    last_message = state["messages"][-1].content

    if "Final Answer:" in last_message:
        return END
    elif "Action:" in last_message:
        return "tool"
    else:
        return END

# ---------------------------
# 7. Build Graph
# ---------------------------
graph = StateGraph(AgentState)

graph.add_node("agent", agent_node)
graph.add_node("tool", tool_node)

graph.set_entry_point("agent")

graph.add_conditional_edges(
    "agent",
    should_continue,
    {
        "tool": "tool",
        END: END,
    },
)

graph.add_edge("tool", "agent")

app = graph.compile()

# ---------------------------
# 8. Runtime Chat Loop
# ---------------------------
if __name__ == "__main__":
    print("ReAct Agent (type 'exit' to quit)\n")

    state = {"messages": []}

    while True:
        user_input = input("You: ")

        if user_input.lower() == "exit":
            print("Goodbye!")
            break

        # Add user message
        state["messages"].append(HumanMessage(content=user_input))

        # Run agent
        result = app.invoke(state)

        # Update state (keeps memory)
        state = result

        # Print latest response
        print("\nAgent:")
        print(state["messages"][-1].content)
        print("\n" + "-" * 50 + "\n")

ReAct Agent (type 'exit' to quit)



You:   What is 15 * 4 + 2?



Agent:
**ReAct Agent Mode Activated**

**Thought:** Hmm, let me think about this multiplication and addition problem... If I multiply 15 by 4, that's a big number! And then adding 2 to it will make it even bigger!

**Action:** Calculator
**Action Input:** 15 * 4 + 2

**Final Answer:** 62

--------------------------------------------------



You:  What is 25 * 4 + 10?



Agent:
**ReAct Agent Mode Activated**

**Thought:** Okay, let me think about this multiplication and addition problem... If I multiply 25 by 4, that's a pretty big number! And then adding 10 to it will make it even bigger!

**Action:** Calculator
**Action Input:** 25 * 4 + 10

**Final Answer:** 110

--------------------------------------------------



You:  exit


Goodbye!
